In [26]:
import re
import numpy as np
import pandas as pd

In [27]:
input_path = "../data/processed/extracted/2017.csv"
output_path = "../data/processed/2017_cleaned.csv"

In [28]:
# 1. Load dataset and clean column headers
df = pd.read_csv(input_path)
df.columns = [re.sub(r"<[^>]+>", "", col).replace("\xa0", " ").strip() for col in df.columns]
print(df.head(5))

                                  #  Are you self-employed?  \
0  e49fe87572831232dcfa51b376b22039                       0   
1  a1eede444ac024928d247a8372d54931                       0   
2  37d3fd67f62bd1e0a2dea4f9cd440d98                       0   
3  519b759442c1cab0e9b5a8a1acb1b216                       0   
4  ef0af4927b575b1a3e607c11ca37870e                       1   

  How many employees does your company or organization have?  \
0                                            100-500           
1                                            100-500           
2                                               6-25           
3                                     More than 1000           
4                                                NaN           

   Is your employer primarily a tech company/organization?  \
0                                                1.0         
1                                                1.0         
2                                                1

In [29]:
# 2. Drop exact duplicates and filter valid ages (14–100)
df = df.drop_duplicates().copy()
age_col = "What is your age?"
df = df[df[age_col].between(14, 100)].copy()
df[age_col] = df[age_col].astype(int)

In [30]:
# 3. Parse date/time columns
for dt_col in ["Start Date (UTC)", "Submit Date (UTC)"]:
    if dt_col in df.columns:
        df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")

In [31]:
# 4. Clean free-text gender responses
gender_col = "What is your gender?"
male_vals = {
    "male", "m", "man", "cis-male", "cis male", "male (cis)", "male, cis",
    "cis hetero male", "mail", "dude",
    "male (hey this is the tech industry you're talking about)"
}
female_vals = {
    "female", "f", "woman", "female (cisgender)", "female (cis)",
    "f, cisgender", "cis female", "cis-female", "my sex is female.",
    "femalw", "femail"
}
non_gender_vals = {"\\-", "contextual", "sometimes", "god king of the valajar"}

def clean_gender(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in male_vals:
        return "male"
    if s in female_vals:
        return "female"
    if s in non_gender_vals:
        return np.nan
    return "other"

df[gender_col] = df[gender_col].map(clean_gender)

In [32]:
ORDER_EMP = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]

emp_code_map = {
    "1-5": 1,
    "6-25": 2,
    "26-100": 3,
    "100-500": 4,
    "500-1000": 5,
    "More than 1000": 6,
}

emp_col = "How many employees does your company or organization have?"

# 1. Make company size an ordered categorical column
df[emp_col] = pd.Categorical(df[emp_col], categories=ORDER_EMP, ordered=True)

# 2. Add numeric indicator column (keeps NaN where company size is missing)
df["no_employees_code"] = df[emp_col].map(emp_code_map).astype("Int64")

# Verify mapping
print(df[[emp_col, "no_employees_code"]].value_counts(dropna=False).sort_index())

How many employees does your company or organization have?  no_employees_code
1-5                                                         1                     20
                                                            2                      0
                                                            3                      0
                                                            4                      0
                                                            5                      0
                                                            6                      0
                                                            <NA>                   0
6-25                                                        1                      0
                                                            2                     86
                                                            3                      0
                                                            4           

In [33]:
# 5. Convert 0/1 binary columns to Yes/No and trim whitespace on text columns
binary_cols = [
    "Are you self-employed?",
    "Is your employer primarily a tech company/organization?",
    "Is your primary role within your company related to tech/IT?",
    "Do you have previous employers?",
    "Have you ever been diagnosed with a mental health disorder?",
    "Have you had a mental health disorder in the past?",
    "Have you ever sought treatment for a mental health disorder from a mental health professional?",
    "Are you openly identified at work as a person with a mental health issue?",
]
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].replace({0: "No", 1: "Yes", 0.0: "No", 1.0: "Yes"})

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

C:\Users\chami\AppData\Local\Temp\ipykernel_2036\3671537303.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [34]:
# 6. Save standalone cleaned 2017 dataset
df = df.reset_index(drop=True)
df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", df.shape)
print(df[age_col].describe())

Saved: ../data/processed/2017_cleaned.csv
Shape: (754, 124)
count    754.000000
mean      34.988064
std        8.338051
min       18.000000
25%       29.000000
50%       34.000000
75%       40.000000
max       67.000000
Name: What is your age?, dtype: float64


In [35]:
print(df.head(5))

                                  # Are you self-employed?  \
0  e49fe87572831232dcfa51b376b22039                     No   
1  a1eede444ac024928d247a8372d54931                     No   
2  37d3fd67f62bd1e0a2dea4f9cd440d98                     No   
3  519b759442c1cab0e9b5a8a1acb1b216                     No   
4  ef0af4927b575b1a3e607c11ca37870e                    Yes   

  How many employees does your company or organization have?  \
0                                            100-500           
1                                            100-500           
2                                               6-25           
3                                     More than 1000           
4                                                NaN           

  Is your employer primarily a tech company/organization?  \
0                                                Yes        
1                                                Yes        
2                                                Yes       